In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
import os

TRAIN_MODE     = True
INFERENCE_MODE = False

KAGGLE = os.path.exists('/kaggle')

DATA_ROOT = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction") if KAGGLE else Path("../data")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

MODEL_SAVE_PATH = Path("saved/lgbm/lgbm_model.txt")  # local
MODEL_LOAD_PATH = Path("/kaggle/input/models/mateomangialomini/lgbm/pytorch/default/11/lgbm_model.txt") if KAGGLE else MODEL_SAVE_PATH


In [ ]:
def load_well(well_dir: Path, well_name: str, is_train: bool):
    hw = pd.read_csv(well_dir / f"{well_name}__horizontal_well.csv")
    tw = pd.read_csv(well_dir / f"{well_name}__typewell.csv")
    hw["WELLNAME"] = well_name
    hw["is_train"] = is_train
    return hw, tw

def load_all(well_dir: Path, is_train: bool):
    wells = {}
    for f in well_dir.glob("*__horizontal_well.csv"):
        name = f.stem.replace("__horizontal_well", "")
        hw, tw = load_well(well_dir, name, is_train)
        wells[name] = (hw, tw)
    return wells

train_wells = load_all(TRAIN_DIR, is_train=True)
test_wells  = load_all(TEST_DIR,  is_train=False)

{'000d7d20': (           MD           X           Y        Z     ANCC    ASTNU    ASTNL  \
0     11467.0  2983525.16  1069022.09 -9258.57 -9395.81 -9569.86 -9597.64   
1     11468.0  2983525.18  1069022.30 -9259.55 -9395.75 -9569.80 -9597.58   
2     11469.0  2983525.20  1069022.52 -9260.52 -9395.69 -9569.74 -9597.52   
3     11470.0  2983525.22  1069022.73 -9261.50 -9395.64 -9569.69 -9597.47   
4     11471.0  2983525.25  1069022.95 -9262.47 -9395.58 -9569.63 -9597.41   
...       ...         ...         ...      ...      ...      ...      ...   
5273  16740.0  2983472.86  1074036.57 -9635.13 -9271.09 -9445.14 -9472.92   
5274  16741.0  2983472.92  1074037.57 -9635.11 -9271.07 -9445.12 -9472.90   
5275  16742.0  2983472.98  1074038.56 -9635.09 -9271.05 -9445.10 -9472.88   
5276  16743.0  2983473.04  1074039.56 -9635.07 -9271.03 -9445.08 -9472.86   
5277  16744.0  2983473.10  1074040.56 -9635.05 -9271.00 -9445.05 -9472.83   

        EGFDU    EGFDL     BUDA       TVT          GR  TVT_in

In [ ]:
def engineer(hw: pd.DataFrame, tw: pd.DataFrame) -> pd.DataFrame:
    hw = hw.sort_values("MD").reset_index(drop=True)
    tw = tw.sort_values("TVT")

    # Last known TVT and its MD position (anchor point)
    known_mask = hw["TVT_input"].notna()
    if known_mask.any():
        last_known_md  = hw.loc[known_mask, "MD"].iloc[-1]
        last_known_tvt = hw.loc[known_mask, "TVT_input"].iloc[-1]
    else:
        last_known_md  = hw["MD"].iloc[0]
        last_known_tvt = 0.0

    hw["last_known_tvt"] = last_known_tvt
    hw["dist_from_anchor"] = hw["MD"] - last_known_md  # 0 in known zone, >0 in eval

    # Typewell GR at last known TVT anchor
    hw["TW_GR_anchor"] = float(np.interp(last_known_tvt, tw["TVT"], tw["GR"]))

    # Typewell GR lookup using TVT_input where known, else extrapolate linearly from anchor
    tvt_ref = hw["TVT_input"].copy()
    # For eval zone: estimate TVT assuming ~same dTVT/dMD as last 20 known rows
    if known_mask.sum() >= 2:
        last_20 = hw.loc[known_mask].tail(20)
        slope = np.polyfit(last_20["MD"], last_20["TVT_input"], 1)[0]
    else:
        slope = 0.0
    hw["tvt_slope_prior"] = slope
    tvt_ref[~known_mask] = last_known_tvt + slope * (hw.loc[~known_mask, "MD"] - last_known_md)
    
    hw["TW_GR"] = np.interp(tvt_ref, tw["TVT"], tw["GR"])
    hw["GR_residual"] = hw["GR"] - hw["TW_GR"]

    # Rolling GR (within-well only — already sorted)
    for w in [5, 20, 50]:
        hw[f"GR_roll{w}"] = hw["GR"].rolling(w, min_periods=1).mean()

    hw["dMD"] = hw["MD"].diff().fillna(0)
    hw["dZ"]  = hw["Z"].diff().fillna(0)
    hw["inc"] = (hw["dZ"] / hw["dMD"].replace(0, np.nan)).ffill().fillna(0)

    return hw
def build_dataset(wells: dict):
    frames = []
    for name, (hw, tw) in wells.items():
        frames.append(engineer(hw, tw))
    return pd.concat(frames, ignore_index=True)

In [ ]:
FEATURES = [
    "MD", "X", "Y", "Z", "GR",
    "last_known_tvt", "dist_from_anchor", "tvt_slope_prior",
    "TW_GR", "TW_GR_anchor", "GR_residual",
    "GR_roll5", "GR_roll20", "GR_roll50",
    "dMD", "dZ", "inc",
    "ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA",
]

if TRAIN_MODE:
    df = build_dataset(train_wells)
    FEATURES = [c for c in FEATURES if c in df.columns]

    train_mask = df["TVT"].notna() & df["TVT_input"].notna()
    eval_mask  = df["TVT"].notna() & df["TVT_input"].isna()

    X_tr, y_tr = df.loc[train_mask, FEATURES], df.loc[train_mask, "TVT"]
    X_ev, y_ev = df.loc[eval_mask,  FEATURES], df.loc[eval_mask,  "TVT"]

    model = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_ev, y_ev)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )

    from sklearn.metrics import root_mean_squared_error
    preds_val = model.predict(X_ev)
    print(f"Val RMSE: {root_mean_squared_error(y_ev, preds_val):.4f}")

    model.booster_.save_model(str(MODEL_SAVE_PATH))
    print(f"Model saved → {MODEL_SAVE_PATH}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006413 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5517
[LightGBM] [Info] Number of data points in the train set: 1308266, number of used features: 23
[LightGBM] [Info] Start training from score 11382.136021
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 3876.6
[200]	valid_0's l2: 2262.07
[300]	valid_0's l2: 2131.92
[400]	valid_0's l2: 2100.35
[500]	valid_0's l2: 2071.67
[600]	valid_0's l2: 2051.77
[700]	valid_0's l2: 2035.69
[800]	valid_0's l2: 2018.35
[900]	valid_0's l2: 2000.87
[1000]	valid_0's l2: 1988.73
[1100]	valid_0's l2: 1980.12
[1200]	valid_0's l2: 1972.1
[1300]	valid_0's l2: 1964.95
[1400]	valid_0's l2: 1958.72
[1500]	valid_0's l2: 1952.08
[1600]	valid_0's l2: 1947.87
[1700]	valid_0's l2: 1943.06
[1800]	valid_0's l2: 1939.63
[1900]	valid_0's

In [ ]:
sample_sub = pd.read_csv(DATA_ROOT / "sample_submission.csv")

if INFERENCE_MODE:
    booster = lgb.Booster(model_file=str(MODEL_LOAD_PATH))
    FEATURES = booster.feature_name()

    df_test   = build_dataset(test_wells)
    test_eval = df_test[df_test["TVT_input"].isna()].copy()

    for c in FEATURES:
        if c not in test_eval.columns:
            test_eval[c] = 0

    test_eval["tvt_pred"] = booster.predict(test_eval[FEATURES])
    test_eval["id"] = test_eval["WELLNAME"] + "_" + test_eval.index.astype(str)

    submission = test_eval[["id", "tvt_pred"]].rename(columns={"tvt_pred": "tvt"})
    submission = sample_sub[["id"]].merge(submission, on="id", how="left")
    submission["tvt"] = submission["tvt"].fillna(submission["tvt"].mean())
    submission.to_csv("submission.csv", index=False)
    print(submission.head())
